# Week 3, Notebook 3: Time Series and Stationarity

**When today depends on yesterday.**

*Author: The Genius Project Year 3*

---

### What you will build
- The skill of reading a **price chart** the way an analyst does.
- A hands-on feel for **stationarity**, the single most important word in
  time-series work.
- The standard fix that turns a wild price into something a model can actually learn.

### The big idea
A **time series** is data with a clock attached: one value per day, in order. Prices
drift, trend, and remember their past, which makes them hard to model directly. The
job of this notebook is to spot *why* they are hard, and to learn the trick that makes
them easy.

### The data: a made-up company

We leave the budget behind and meet **Coral Reef Games** (ticker `REEF`), a
fictional company. The file `data/daily_stock.csv` holds about three years of daily
trading, one row per weekday.

| Column | What it means |
| --- | --- |
| `date` | The trading day. |
| `price` | The share price at the end of that day, in dollars. |
| `volume` | How many shares changed hands that day. |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

stock = pd.read_csv("data/daily_stock.csv", parse_dates=["date"])
stock = stock.set_index("date")  # put the clock on the index

print("Trading days:", len(stock))
print(f"First price: ${stock['price'].iloc[0]:.2f}   Last price: ${stock['price'].iloc[-1]:.2f}")

plt.figure(figsize=(10, 4))
plt.plot(stock.index, stock["price"], color="#4C72B0")
plt.title("Coral Reef Games (REEF) - daily price")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.grid(alpha=0.3)
plt.show()

### Step 1: What "stationary" means

A time series is **stationary** when its behaviour does not depend on *when* you look:
the average stays put and the size of the wobble stays about the same over time. A
**non-stationary** series drifts, its average today is nowhere near its average
a year ago.

Why care? Almost every forecasting tool assumes the future will behave like the past.
If the average keeps moving, "the past" is a moving target and the tool gets fooled.
The price chart above clearly trends upward, so we already suspect it is **not**
stationary. Let's prove it.

### Step 2: Watch the average move (rolling statistics)

A **rolling mean** is the average of the last 30 days, recomputed each day. A
**rolling standard deviation** is the wobble over the same window. If the series were
stationary, both lines would stay flat. Watch the rolling mean climb.

In [ ]:
roll_mean = stock["price"].rolling(30).mean()
roll_std = stock["price"].rolling(30).std()

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(stock.index, stock["price"], color="#c9c9c9", label="daily price")
ax[0].plot(roll_mean.index, roll_mean, color="#C44E52", label="30-day average")
ax[0].set_ylabel("Price ($)")
ax[0].set_title("The 30-day average keeps climbing - not stationary")
ax[0].legend()

ax[1].plot(roll_std.index, roll_std, color="#8172B3")
ax[1].set_ylabel("30-day wobble ($)")
ax[1].set_title("The size of the swings drifts too")
ax[1].set_xlabel("Date")
plt.tight_layout()
plt.show()

### Step 3: The fix, look at daily *returns*, not prices

Instead of the price itself, analysts study the **return**: the percent change from
one day to the next. This is called **differencing**, and it is the workhorse fix for
non-stationary data. A price of \$60 to \$61 becomes a return of about +1.7%.

Returns throw away the "how high is the price" information (the part that drifts) and
keep only "how much did it move today" (the part that stays steady).

In [ ]:
stock["return"] = stock["price"].pct_change()  # today vs yesterday, as a fraction

plt.figure(figsize=(10, 4))
plt.plot(stock.index, stock["return"], color="#55A868", linewidth=0.7)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Daily returns wobble around zero - this looks stationary")
plt.xlabel("Date")
plt.ylabel("Daily return")
plt.show()

In [ ]:
# The proof: the rolling average of returns sits flat near zero,
# instead of climbing the way the price did.
ret_roll_mean = stock["return"].rolling(30).mean()

plt.figure(figsize=(10, 3.5))
plt.plot(ret_roll_mean.index, ret_roll_mean, color="#C44E52")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("30-day average of returns - flat and steady (stationary)")
plt.xlabel("Date")
plt.ylabel("Average return")
plt.show()

### Step 4: Does yesterday predict today? (autocorrelation)

**Autocorrelation** measures how much a series is related to its own past. We compare
each day with the day `lag` steps before it. A value near **1** means "the past
predicts the present"; near **0** means "no memory".

We will compute it for the price and for the returns, and the contrast is the whole
lesson.

In [ ]:
def autocorrelation(series, lag):
    "How closely a series follows its own value `lag` days ago (between -1 and 1)."
    s = series.dropna().values
    return np.corrcoef(s[:-lag], s[lag:])[0, 1]

lags = [1, 2, 5, 10, 20]
price_ac = [autocorrelation(stock["price"], k) for k in lags]
return_ac = [autocorrelation(stock["return"], k) for k in lags]

print("lag | price memory | return memory")
for k, p, r in zip(lags, price_ac, return_ac):
    print(f"{k:>3} | {p:>11.2f} | {r:>12.2f}")

x = np.arange(len(lags))
plt.figure(figsize=(9, 4))
plt.bar(x - 0.2, price_ac, width=0.4, label="price", color="#4C72B0")
plt.bar(x + 0.2, return_ac, width=0.4, label="return", color="#55A868")
plt.xticks(x, lags)
plt.title("The price remembers its past; the returns barely do")
plt.xlabel("Lag (days back)")
plt.ylabel("Autocorrelation")
plt.axhline(0, color="black", linewidth=0.8)
plt.legend()
plt.show()

The price bars are near 1: today's price is almost yesterday's price, so it carries a
strong trend (non-stationary). The return bars are near 0: knowing today's move tells
you almost nothing about tomorrow's. That near-zero memory is exactly why beating the
market is so hard, a lesson we cash in during Notebook 5.

### What you learned
- A **time series** is ordered-in-time data; today can depend on yesterday.
- **Stationary** means the average and the wobble stay put over time.
- Prices are **non-stationary** (they drift); **returns** are roughly stationary.
- **Differencing** (turning prices into returns) is the standard fix.
- **Autocorrelation** measures a series' memory of its own past.

### Try it yourself
- Change the rolling window from `30` to `90` days. Does the average look smoother?
- Compute the autocorrelation of `volume`. Does trading activity have memory?

### Next
Now that we can tame a price into something steady, Notebook 4 does the exciting part:
**forecasting**, and measuring, honestly, how good (or bad) our guesses are.